In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load all files
train = pd.read_csv('../data/train.csv')
stores = pd.read_csv('../data/stores.csv')
features = pd.read_csv('../data/features.csv')

print("Train shape:", train.shape)
print("Stores shape:", stores.shape)
print("Features shape:", features.shape)

Train shape: (421570, 5)
Stores shape: (45, 3)
Features shape: (8190, 12)


In [3]:
print("=== TRAIN ===")
print(train.head())
print("\nData Types:")
print(train.dtypes)
print("\nMissing Values:")
print(train.isnull().sum())

print("\n=== STORES ===")
print(stores.head())

print("\n=== FEATURES ===")
print(features.head())
print("\nMissing Values in Features:")
print(features.isnull().sum())

=== TRAIN ===
   Store  Dept        Date  Weekly_Sales  IsHoliday
0      1     1  2010-02-05      24924.50      False
1      1     1  2010-02-12      46039.49       True
2      1     1  2010-02-19      41595.55      False
3      1     1  2010-02-26      19403.54      False
4      1     1  2010-03-05      21827.90      False

Data Types:
Store             int64
Dept              int64
Date             object
Weekly_Sales    float64
IsHoliday          bool
dtype: object

Missing Values:
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

=== STORES ===
   Store Type    Size
0      1    A  151315
1      2    A  202307
2      3    B   37392
3      4    A  205863
4      5    B   34875

=== FEATURES ===
   Store        Date  Temperature  Fuel_Price  MarkDown1  MarkDown2  \
0      1  2010-02-05        42.31       2.572        NaN        NaN   
1      1  2010-02-12        38.51       2.548        NaN        NaN   
2      1  2010-02-19        

In [4]:
# Fix date columns
train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])

# Merge all three files into one master dataframe
df = train.merge(stores, on='Store', how='left')
df = df.merge(features, on=['Store', 'Date'], how='left')

# Fill missing markdown values with 0 (no markdown = no promotion)
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df[markdown_cols] = df[markdown_cols].fillna(0)

# Drop duplicate IsHoliday column created during merge
df = df.drop(columns=['IsHoliday_y'])
df = df.rename(columns={'IsHoliday_x': 'IsHoliday'})

print("Master dataframe shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDate range:", df['Date'].min(), "to", df['Date'].max())

Master dataframe shape: (421570, 16)

Columns: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']

Missing values:
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
Type            0
Size            0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
dtype: int64

Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00


In [5]:
# Save cleaned master file for SQL phase
df.to_csv('../data/cleaned_data.csv', index=False)
print("Cleaned data saved successfully")

# Quick summary stats
print("\nWeekly Sales Summary:")
print(df['Weekly_Sales'].describe())

Cleaned data saved successfully

Weekly Sales Summary:
count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000
Name: Weekly_Sales, dtype: float64
